# Book N-Gram Feature Extractor
Extracts the top 100 unigrams, bigrams, and trigrams from each book `.txt` file.
Used for genre classification feature engineering.

In [10]:
import re
import os
import pandas as pd
from collections import Counter
from nltk.util import ngrams
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

True

## Configuration
Edit the `BOOKS` dictionary to match your `.txt` filenames and genres.
Place all `.txt` files in the same folder as this notebook (or update `BOOKS_DIR`).

In [11]:
# --- Configuration ---
BOOKS_DIR = "."  # Folder containing the .txt files
TOP_N     = 100  # Number of top n-grams to extract

BOOKS = {
    "TheWonderfulWizardOfOz":          "fantasy",
    # "TheGodsOfPegana":                 "fantasy",
    # "TheEnchantedCastle":              "fantasy",
    # "TheKingOfElflandsDaughter":       "fantasy",

    "PrideAndPrejudice":                "romance",
    # "JaneEyre":                         "romance",
    # "ARoomWithAView":                   "romance",
    # "TheBlueCastle":                    "romance",

    "OnTheTrailOfTheSpacePirates":      "sci-fi",
    # "TheTimeMachine":                   "sci-fi",
    # "APrincessOfMars":                  "sci-fi",
    # "TwentyThousandLeagues":            "sci-fi",

    "Frankenstein":                     "horror",
    # "Dracula":                          "horror",
    # "JekyllAndHyde":                    "horror",
    # "KingInYellow":                     "horror",

    "AutobiographyOfBenjaminFranklin":  "autobiography",
    # "LifeOnTheMississippi":             "autobiography",
    # "NarrativeOfFrederickDouglass":     "autobiography",
    # "ConfessionsOfStAugustine":         "autobiography"
}

## Clean Raw Text Files (Project Gutenberg)
Strips the standard Project Gutenberg header and footer boilerplate from each `.txt` file
and saves cleaned versions as `<BookName>_clean.txt` in the same directory.
Run this once before extracting n-grams.

In [12]:
import re
import os

# Project Gutenberg delimiter patterns
START_PATTERN = re.compile(r'\*{3}\s*START OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}', re.IGNORECASE)
END_PATTERN   = re.compile(r'\*{3}\s*END OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}',   re.IGNORECASE)

def clean_gutenberg(filename):
    """
    Strip Project Gutenberg header and footer from a .txt file.
    Saves the cleaned text as <filename>_clean.txt.
    Returns the cleaned text as a string.
    """
    path = os.path.join(BOOKS_DIR, filename + '.txt')
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    # Find the start marker
    start_match = START_PATTERN.search(raw)
    if start_match:
        text = raw[start_match.end():]
    else:
        print(f'  [WARNING] No START marker found in {filename} — using full text')
        text = raw

    # Find the end marker
    end_match = END_PATTERN.search(text)
    if end_match:
        text = text[:end_match.start()]
    else:
        print(f'  [WARNING] No END marker found in {filename} — keeping text until EOF')

    text = text.strip()

    # Save cleaned version
    out_path = os.path.join(BOOKS_DIR, filename + '_clean.txt')
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(text)

    original_words = len(raw.split())
    cleaned_words  = len(text.split())
    removed_words  = original_words - cleaned_words
    print(f'  {filename}: {original_words:,} → {cleaned_words:,} words  (removed {removed_words:,} boilerplate words)')
    return text

print('Cleaning Project Gutenberg boilerplate...\n')
for book in BOOKS:
    clean_gutenberg(book)

print('\nCleaned files saved as <BookName>_clean.txt')

Cleaning Project Gutenberg boilerplate...

  TheWonderfulWizardOfOz: 42,692 → 39,649 words  (removed 3,043 boilerplate words)
  PrideAndPrejudice: 130,415 → 127,359 words  (removed 3,056 boilerplate words)
  OnTheTrailOfTheSpacePirates: 55,595 → 52,524 words  (removed 3,071 boilerplate words)
  Frankenstein: 78,106 → 75,042 words  (removed 3,064 boilerplate words)
  AutobiographyOfBenjaminFranklin: 79,264 → 76,203 words  (removed 3,061 boilerplate words)

Cleaned files saved as <BookName>_clean.txt


## Helper Functions

In [13]:
STOP_WORDS = set(stopwords.words('english'))

# Custom stop words based on top unigrams from each book, to remove character names and other genre-specific terms that might skew the analysis
CUSTOM_STOPWORDS = {'dorothy', 'toto', 'scarecrow', 'tin', 'woodman', 'mr', 'bingley',
                    'elizabeth', 'darcy', 'bennet', 'justine', 'clerval', 'victor', 'mrs', 
                    'franklin', 'astro', 'roger', 'tom', 'coxine', 'lion', 'wallace',
                    'philadelphia', 'oz', 'kansas', 'illustration', 'pennsylvania',
                    'london', 'could', 'would', 'jane'}

# Custom stop words for 20-book training set
""" CUSTOM_STOPWORDS = {'dorothy', 'toto', 'scarecrow', 'tin', 'woodman', 'mr', 'bingley',
                    'elizabeth', 'darcy', 'bennet', 'justine', 'clerval', 'victor', 'mrs', 
                    'franklin', 'astro', 'roger', 'tom', 'coxine', 'lion', 'wallace',
                    'philadelphia', 'oz', 'kansas', 'illustration', 'pennsylvania', 'london', 
                    'anthea', 'robert', 'cyril', 'jane', 'rochester', 'lucy', 'cecil', 'beebe',
                    'bartlett', 'honeychurch', 'george', 'freddy', 'emerson', 'charlotte',
                    'utterson', 'jekyll', 'hyde', 'clifford'} """

def load_text(filename):
    """Read a .txt file and return its contents as a string."""
    path = os.path.join(BOOKS_DIR, filename + "_clean.txt")
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def tokenize(text):
    """Lowercase, strip punctuation, remove stop words."""
    words = re.findall(r'[a-z]+', text.lower())
    return [w for w in words if w not in STOP_WORDS and w not in CUSTOM_STOPWORDS and len(w) > 1]
    #return [w for w in words if w not in STOP_WORDS and len(w) > 1]

def top_ngrams(tokens, n, top_n=TOP_N):
    """Return the top_n most common n-grams as a list of (ngram_string, count) tuples."""
    counts = Counter(ngrams(tokens, n))
    return [(' '.join(gram), count) for gram, count in counts.most_common(top_n)]

## Extract N-Grams for All Books

In [14]:
results = {}  # { book_name: { 'genre': ..., 'unigrams': [...], 'bigrams': [...], 'trigrams': [...] } }

for book, genre in BOOKS.items():
    print(f"Processing: {book} ({genre})...")
    text   = load_text(book)
    tokens = tokenize(text)

    results[book] = {
        'genre':    genre,
        'unigrams': top_ngrams(tokens, 1),
        'bigrams':  top_ngrams(tokens, 2),
        'trigrams': top_ngrams(tokens, 3),
    }

print("\nDone!")

Processing: TheWonderfulWizardOfOz (fantasy)...
Processing: PrideAndPrejudice (romance)...
Processing: OnTheTrailOfTheSpacePirates (sci-fi)...
Processing: Frankenstein (horror)...
Processing: AutobiographyOfBenjaminFranklin (autobiography)...

Done!


## Display Results per Book

In [15]:
def show_book(book_name):
    """Pretty-print the top n-grams for a single book."""
    data = results[book_name]
    print(f"\n{'='*60}")
    print(f"  {book_name}  [{data['genre'].upper()}]")
    print(f"{'='*60}")

    for label, key in [("TOP 100 UNIGRAMS", 'unigrams'),
                       ("TOP 100 BIGRAMS",  'bigrams'),
                       ("TOP 100 TRIGRAMS", 'trigrams')]:
        print(f"\n--- {label} ---")
        df = pd.DataFrame(data[key], columns=['ngram', 'count'])
        df.index += 1
        display(df)

# Show all books
for book in BOOKS:
    show_book(book)


  TheWonderfulWizardOfOz  [FANTASY]

--- TOP 100 UNIGRAMS ---


,ngram,count
1,said,332
2,great,142
3,little,139
4,witch,129
5,one,126
...,...,...
96,nothing,30
97,also,30
98,terrible,29
99,tree,29



--- TOP 100 BIGRAMS ---


,ngram,count
1,wicked witch,60
2,emerald city,57
3,little girl,32
4,winged monkeys,30
5,aunt em,24
...,...,...
96,upon head,5
97,kill wicked,5
98,great head,5
99,witch said,5



--- TOP 100 TRIGRAMS ---


...

,ngram,count
1,one,294
2,time,201
3,great,174
4,good,160
5,made,155
...,...,...
96,brought,46
97,service,45
98,done,45
99,things,45



--- TOP 100 BIGRAMS ---


,ngram,count
1,new york,38
2,printing house,28
3,poor richard,25
4,new england,20
5,good deal,18
...,...,...
96,union colonies,4
97,abraham speech,4
98,first part,4
99,old friend,4



--- TOP 100 TRIGRAMS ---


,ngram,count
1,poor richard says,11
2,poor richard almanac,9
3,new england courant,7
4,new printing office,5
5,father abraham speech,4
...,...,...
96,america said doctor,2
97,said doctor thomas,2
98,doctor thomas jefferson,2
99,henry holt company,2


## Quick Comparison: Top 10 Unigrams Side-by-Side
A single table showing the most distinctive words per genre at a glance.

In [16]:
comparison = {}
for book, data in results.items():
    label = f"{book}\n({data['genre']})"
    comparison[label] = [ngram for ngram, _ in data['unigrams'][:10]]

comp_df = pd.DataFrame(comparison)
comp_df.index = [f"#{i+1}" for i in range(10)]
print("Top 10 unigrams per book (stop words removed):\n")
display(comp_df)

Top 10 unigrams per book (stop words removed):



,TheWonderfulWizardOfOz\n(fantasy),PrideAndPrejudice\n(romance),OnTheTrailOfTheSpacePirates\n(sci-fi),Frankenstein\n(horror),AutobiographyOfBenjaminFranklin\n(autobiography)
#1,said,said,strong,one,one
#2,great,much,said,yet,time
#3,little,miss,ship,man,great
#4,witch,must,space,father,good
#5,one,one,one,upon,made
#6,asked,know,sir,life,little
#7,green,though,solar,every,much
#8,came,well,two,first,might
#9,good,never,get,might,first
#10,back,think,turned,shall,house


## Export Results to CSV
One CSV per n-gram type, with a column for each book.

In [17]:
for ngram_type in ['unigrams', 'bigrams', 'trigrams']:
    rows = []
    for book, data in results.items():
        for rank, (ngram, count) in enumerate(data[ngram_type], start=1):
            rows.append({
                'book':       book,
                'genre':      data['genre'],
                'rank':       rank,
                'ngram':      ngram,
                'count':      count,
            })
    df = pd.DataFrame(rows)
    out_path = f"top100_{ngram_type}.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

Saved: top100_unigrams.csv
Saved: top100_bigrams.csv
Saved: top100_trigrams.csv


## Export Vocabulary for Classifier
Saves `vocabulary.csv` — the union of all top-100 n-grams across every book,
de-duplicated. The classifier notebook uses this list to build its feature matrix.

In [18]:
# Build the union vocabulary from all books' top-100 unigrams
all_unigrams = set()
for data in results.values():
    for word, _ in data['unigrams']:
        all_unigrams.add(word)
    # Include bigrams and trigrams as well, since they will be used in the TF-IDF N-grams vectorizer
    for word, _ in data['bigrams']:
        all_unigrams.add(word)
    for word, _ in data['trigrams']:
        all_unigrams.add(word)

vocab_df = pd.DataFrame(sorted(all_unigrams), columns=['word'])
vocab_df.to_csv('vocabulary.csv', index=False)
print(f"Vocabulary size: {len(vocab_df)} unique n-grams")
print(f"Saved to 'vocabulary.csv'")


Vocabulary size: 1259 unique n-grams
Saved to 'vocabulary.csv'
